In [1]:
! pip install mlflow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


In [2]:
! pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


In [3]:
! pip install sickit-learn

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement sickit-learn (from versions: none)
ERROR: No matching distribution found for sickit-learn

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


In [20]:
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# ==========================
# Load Dataset
# ==========================
digits = datasets.load_digits()

X = digits.data
y = digits.target

print("Dataset Shape:", X.shape)
print("Number of Classes:", len(np.unique(y)))

# ==========================
# Split Dataset
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# ==========================
# Feature Scaling
# ==========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================
# Start MLflow Run
# ==========================
with mlflow.start_run(run_name="SVM_Digits"):

    # Log Parameters
    mlflow.log_param("Model", "SVM")
    mlflow.log_param("Kernel", "rbf")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("Gamma", "scale")
    mlflow.log_param("Train Size", len(X_train))
    mlflow.log_param("Test Size", len(X_test))
    mlflow.log_artifact("requirements.txt")
    # ==========================
    # Train Model
    # ==========================
    svm = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        random_state=42
    )

    svm.fit(X_train, y_train)

    # ==========================
    # Prediction
    # ==========================
    y_pred = svm.predict(X_test)

    # ==========================
    # Evaluation
    # ==========================
    accuracy = accuracy_score(y_test, y_pred)

    print(f"\nAccuracy = {accuracy:.4f}")

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Log Metrics
    mlflow.log_metric("Accuracy", accuracy)

    # Log Classification Report
    report = classification_report(y_test, y_pred)
    with open("classification_report.txt", "w") as f:
        f.write(report)

    mlflow.log_artifact("classification_report.txt")
    mlflow.log_artifact("requirements.txt")

    # Log Confusion Matrix Figure
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")

    plt.savefig("confusion_matrix.png")
    plt.close()

    mlflow.log_artifact("confusion_matrix.png")

    # Log Model
    mlflow.sklearn.log_model(
        sk_model=svm,
        artifact_path="svm_model"
    )

print("\nRun completed successfully!")

Dataset Shape: (1797, 64)
Number of Classes: 10

Accuracy = 0.9833

Confusion Matrix:
[[54  0  0  0  0  0  0  0  0  0]
 [ 0 54  0  0  1  0  0  0  0  0]
 [ 0  0 52  0  1  0  0  0  0  0]
 [ 0  0  0 55  0  0  0  0  0  0]
 [ 0  0  0  0 53  0  0  1  0  0]
 [ 0  0  0  0  0 54  0  0  0  1]
 [ 0  0  0  0  0  0 54  0  0  0]
 [ 0  0  0  0  0  0  0 54  0  0]
 [ 0  2  0  0  1  0  0  0 49  0]
 [ 0  0  0  0  0  0  1  1  0 52]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        54
           1       0.96      0.98      0.97        55
           2       1.00      0.98      0.99        53
           3       1.00      1.00      1.00        55
           4       0.95      0.98      0.96        54
           5       1.00      0.98      0.99        55
           6       0.98      1.00      0.99        54
           7       0.96      1.00      0.98        54
           8       1.00      0.94      0.97        52
           9      

2026/08/12 01:09:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Run completed successfully!


In [21]:
import joblib

joblib.dump(svm, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

# Flask Web Application

This section creates an MLflow-style web interface for uploading a digit image and predicting its class with the trained SVM model.

In [22]:
from flask import Flask, request, render_template_string
from PIL import Image, ImageOps, ImageDraw, ImageChops
import numpy as np
import joblib
import os
import base64
from io import BytesIO

app = Flask(__name__)

BASE_DIR = os.getcwd()
MODEL_PATH = os.path.join(BASE_DIR, "model.pkl")
SCALER_PATH = os.path.join(BASE_DIR, "scaler.pkl")

model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

MODEL_ACCURACY = 0.9833
ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "bmp", "webp"}


def allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS


def prepare_digit_image(file_storage):
    """
    Prepare an external digit image for the sklearn Digits SVM.

    Returns:
        original_image: original uploaded image for display
        processed_8x8: final 8x8 image used by the model
        features: scaled 64-feature vector
    """
    original = Image.open(file_storage).convert("RGB")
    gray = ImageOps.grayscale(original)

    arr = np.asarray(gray, dtype=np.uint8)

    # Determine foreground polarity from border/background.
    border = np.concatenate([
        arr[0, :], arr[-1, :], arr[:, 0], arr[:, -1]
    ])
    background_is_bright = border.mean() > 127

    # Convert to white digit on black background, matching sklearn Digits.
    if background_is_bright:
        ink = 255 - arr
    else:
        ink = arr.copy()

    # Remove very weak background noise.
    threshold = max(25, int(np.percentile(ink, 85)))
    mask = ink > threshold

    if not np.any(mask):
        # Fallback: use the complete image.
        crop = Image.fromarray(ink)
    else:
        ys, xs = np.where(mask)
        x0, x1 = xs.min(), xs.max()
        y0, y1 = ys.min(), ys.max()

        # Crop tightly around the digit.
        crop = Image.fromarray(ink[y0:y1 + 1, x0:x1 + 1])

    # Fit digit into a square while preserving aspect ratio.
    w, h = crop.size
    side = max(w, h)
    canvas = Image.new("L", (side, side), 0)

    left = (side - w) // 2
    top = (side - h) // 2
    canvas.paste(crop, (left, top))

    # Add margin. This is important because sklearn Digits contains
    # digits surrounded by background rather than filling the whole 8x8.
    margin = max(2, int(side * 0.18))
    canvas_with_margin = Image.new(
        "L",
        (side + 2 * margin, side + 2 * margin),
        0
    )
    canvas_with_margin.paste(canvas, (margin, margin))

    # Resize to 8x8, then map [0,255] -> [0,16].
    processed_8x8 = canvas_with_margin.resize(
        (8, 8),
        Image.Resampling.LANCZOS
    )

    pixel_values = np.asarray(processed_8x8, dtype=np.float64)
    pixel_values = (pixel_values / 255.0) * 16.0

    features = pixel_values.reshape(1, 64)
    features_scaled = scaler.transform(features)

    return original, processed_8x8, features_scaled


def to_base64(image, size=(360, 360), prediction=None):
    display = image.copy().convert("RGB")
    display.thumbnail(size, Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", size, "white")
    x = (size[0] - display.width) // 2
    y = (size[1] - display.height) // 2
    canvas.paste(display, (x, y))

    if prediction is not None:
        draw = ImageDraw.Draw(canvas)
        draw.rectangle((0, 0, size[0], 50), fill="black")
        draw.text(
            (14, 14),
            f"Prediction: {prediction}",
            fill="white"
        )

    buffer = BytesIO()
    canvas.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Digits Prediction</title>

<style>
* { box-sizing: border-box; }

body {
    margin: 0;
    font-family: Inter, -apple-system, BlinkMacSystemFont,
                 "Segoe UI", Roboto, Arial, sans-serif;
    background: #f7f8fa;
    color: #202124;
}

.layout { display: flex; min-height: 100vh; }

.sidebar {
    width: 250px;
    background: #17191c;
    color: #e8eaed;
    padding: 24px 18px;
    position: fixed;
    inset: 0 auto 0 0;
}

.brand {
    font-size: 21px;
    font-weight: 700;
    margin-bottom: 34px;
    padding: 0 10px;
}

.nav-title {
    color: #9aa0a6;
    font-size: 12px;
    text-transform: uppercase;
    letter-spacing: .08em;
    padding: 0 10px;
    margin-bottom: 10px;
}

.nav-item {
    padding: 11px 12px;
    border-radius: 7px;
    margin-bottom: 5px;
    color: #d7d9dc;
    background: #24272b;
}

.nav-item.active {
    background: #3a3f45;
    color: white;
}

.sidebar-info {
    position: absolute;
    left: 28px;
    right: 28px;
    bottom: 24px;
    color: #9aa0a6;
    font-size: 12px;
    line-height: 1.6;
}

.main {
    margin-left: 250px;
    width: calc(100% - 250px);
    padding: 34px 44px 60px;
}

.header, .content {
    max-width: 1100px;
    margin-left: auto;
    margin-right: auto;
}

.header { margin-bottom: 28px; }

.header h1 {
    margin: 0 0 7px;
    font-size: 28px;
}

.header p {
    margin: 0;
    color: #6b7077;
}

.card {
    background: white;
    border: 1px solid #e2e5e9;
    border-radius: 10px;
    box-shadow: 0 2px 8px rgba(0,0,0,.04);
    padding: 28px;
    margin-bottom: 22px;
}

.card-title {
    font-size: 17px;
    font-weight: 650;
    margin-bottom: 18px;
}

.upload-area {
    border: 2px dashed #c8cdd3;
    border-radius: 10px;
    padding: 48px 20px;
    text-align: center;
}

.upload-area h2 {
    margin: 0 0 8px;
    font-size: 19px;
}

.upload-area p {
    color: #70757d;
    margin: 0 0 20px;
}

input[type=file] { display: none; }

.file-label {
    display: inline-block;
    background: #202124;
    color: white;
    padding: 11px 18px;
    border-radius: 6px;
    cursor: pointer;
    font-weight: 600;
}

.file-name {
    margin-top: 12px;
    color: #5f6368;
    font-size: 13px;
}

.predict-btn {
    margin-top: 18px;
    width: 100%;
    padding: 13px;
    border: 0;
    border-radius: 6px;
    background: #111315;
    color: white;
    font-size: 15px;
    font-weight: 650;
    cursor: pointer;
}

.result-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 24px;
}

.image-card {
    border: 1px solid #e1e4e8;
    border-radius: 8px;
    padding: 14px;
    text-align: center;
}

.image-card img {
    width: 100%;
    max-width: 360px;
    height: 360px;
    object-fit: contain;
    border-radius: 6px;
    border: 1px solid #e1e4e8;
}

.image-title {
    font-size: 14px;
    font-weight: 650;
    margin-bottom: 12px;
}

.processed {
    image-rendering: pixelated;
    background: #111;
}

.result-value {
    font-size: 64px;
    font-weight: 750;
    line-height: 1;
    margin: 10px 0 22px;
}

.metric {
    display: flex;
    justify-content: space-between;
    padding: 13px 0;
    border-bottom: 1px solid #eceef0;
}

.metric-label { color: #70757d; }
.metric-value { font-weight: 650; }

.accuracy-bar {
    height: 9px;
    background: #e9ecef;
    border-radius: 20px;
    overflow: hidden;
    margin-top: 10px;
}

.accuracy-fill {
    height: 100%;
    width: {{ accuracy }}%;
    background: #343a40;
}

.error {
    padding: 14px 16px;
    border-radius: 7px;
    background: #fff1f1;
    color: #b42318;
    border: 1px solid #f3c7c7;
}

.hint {
    color: #7a7f86;
    font-size: 12px;
    margin-top: 12px;
    line-height: 1.5;
}

@media (max-width: 800px) {
    .sidebar { width: 200px; }
    .main {
        margin-left: 200px;
        width: calc(100% - 200px);
        padding: 25px;
    }
    .result-grid { grid-template-columns: 1fr; }
}

@media (max-width: 600px) {
    .sidebar { display: none; }
    .main {
        margin-left: 0;
        width: 100%;
        padding: 18px;
    }
}
</style>
</head>

<body>
<div class="layout">

<aside class="sidebar">
    <div class="brand">Digits ML</div>

    <div class="nav-title">Workspace</div>
    <div class="nav-item active">Prediction</div>
    <div class="nav-item">Model Information</div>
    <div class="nav-item">Evaluation</div>

    <div class="sidebar-info">
        <strong>SVM · RBF</strong><br>
        sklearn Digits Dataset<br>
        10 classes · 8×8 input
    </div>
</aside>

<main class="main">

<div class="header">
    <h1>Digits Prediction</h1>
    <p>Upload a handwritten digit and run inference using the trained SVM model.</p>
</div>

<div class="content">

<div class="card">
    <div class="card-title">Input Image</div>

    <form method="POST" enctype="multipart/form-data">
        <div class="upload-area">
            <h2>Upload a digit image</h2>
            <p>PNG, JPG, JPEG, BMP or WEBP</p>

            <label class="file-label" for="file">Choose image</label>

            <input
                id="file"
                type="file"
                name="file"
                accept=".png,.jpg,.jpeg,.bmp,.webp"
                required
                onchange="document.getElementById('file-name').textContent =
                          this.files[0]?.name || ''"
            >

            <div id="file-name" class="file-name"></div>
        </div>

        <button class="predict-btn" type="submit">
            Predict Digit
        </button>
    </form>

    <div class="hint">
        The original image is preserved for display. A separate preprocessing
        pipeline crops the digit, centers it, adds margin, converts it to 8×8,
        maps pixels to 0–16, and applies the training StandardScaler.
    </div>
</div>

{% if error %}
<div class="card">
    <div class="error">{{ error }}</div>
</div>
{% endif %}

{% if prediction is not none %}
<div class="card">
    <div class="card-title">Prediction Result</div>

    <div class="result-grid">

        <div class="image-card">
            <div class="image-title">Uploaded Image</div>
            <img
                src="data:image/png;base64,{{ original_image }}"
                alt="Original uploaded image"
            >
        </div>

        <div class="image-card">
            <div class="image-title">Processed Image (8×8)</div>
            <img
                class="processed"
                src="data:image/png;base64,{{ processed_image }}"
                alt="Processed image"
            >
        </div>

    </div>

    <div style="margin-top:24px">
        <div class="metric-label">Predicted Class</div>
        <div class="result-value">{{ prediction }}</div>

        <div class="metric">
            <span class="metric-label">Model</span>
            <span class="metric-value">SVM (RBF)</span>
        </div>

        <div class="metric">
            <span class="metric-label">Classes</span>
            <span class="metric-value">0 – 9</span>
        </div>

        <div class="metric">
            <span class="metric-label">Test Accuracy</span>
            <span class="metric-value">{{ "%.2f"|format(accuracy) }}%</span>
        </div>

        <div class="accuracy-bar">
            <div class="accuracy-fill"></div>
        </div>
    </div>
</div>
{% endif %}

</div>
</main>
</div>
</body>
</html>
"""


@app.route("/", methods=["GET", "POST"])
def index():
    prediction = None
    original_image = None
    processed_image = None
    error = None

    if request.method == "POST":
        file = request.files.get("file")

        if file is None or file.filename == "":
            error = "Please choose an image."
        elif not allowed_file(file.filename):
            error = "Unsupported image format."
        else:
            try:
                original, processed, features = prepare_digit_image(file)

                prediction = int(model.predict(features)[0])

                original_image = to_base64(original)
                processed_image = to_base64(
                    processed,
                    size=(360, 360),
                    prediction=prediction
                )

            except Exception as exc:
                error = f"Prediction failed: {exc}"

    return render_template_string(
        HTML,
        prediction=prediction,
        original_image=original_image,
        processed_image=processed_image,
        error=error,
        accuracy=MODEL_ACCURACY * 100
    )


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
